# fly-brain quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victory-c/fly-brain/blob/main/notebooks/00_quickstart.ipynb)

The whole male fruit-fly central nervous system (166,700 neurons, male CNS v1.0) as a spiking network. In this
notebook you will:

1. load the brain and look neurons up by name, side, class and position (`brain.select`);
2. replay the feeding experiment of Shiu et al. 2024: drive the sugar taste neurons and watch MN9, the motor
   neuron that extends the proboscis;
3. silence one cell type and see whether MN9 still responds;
4. plot a spike raster.

**Runtime.** On Colab pick *Runtime > Change runtime type > T4 GPU* first. Without a GPU everything still runs,
with fewer flies and rates, and much more slowly. The first run downloads the prebuilt brain (about 220 MB).

Words you do not know are in [docs/GLOSSARY.md](https://github.com/victory-c/fly-brain/blob/main/docs/GLOSSARY.md).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo():
    """The fly-brain checkout this notebook runs in; on Colab, clone the parts it needs."""
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "fly_brain" / "brain" / "connectome.py").exists():
            return d
    if "google.colab" in sys.modules:
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/victory-c/fly-brain.git"], check=True)
        subprocess.run(["git", "-C", "fly-brain", "sparse-checkout", "set", "fly_brain/brain"], check=True)
        return Path("fly-brain").resolve()
    raise SystemExit("open this notebook from inside a clone of github.com/victory-c/fly-brain")


REPO = find_repo()
os.chdir(REPO / "fly_brain")
sys.path.insert(0, str(REPO / "fly_brain"))
if not all(Path(f).exists() for f in ("brain.npz", "brain_meta.parquet", "web_data/atlas.npz")):
    from brain import fetch
    fetch.fetch()  # the prebuilt brain from the GitHub release, checked against SHA256SUMS

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
print("running on", torch.cuda.get_device_name(0) if GPU else "the CPU (slow: the cells below use fewer flies)")

## 1. The brain

`Brain()` loads the wiring: for every neuron, which neurons it synapses onto and with how many synapses, signed
by its transmitter (GABA, glutamate and histamine inhibit). Neurons are numbered 0 to n-1 in the simulator; that
number is the **row**. The dataset's own id is the **bodyId**. `brain.meta` has one line per row with its
bodyId, cell type, side and transmitter.

In [ ]:
from brain.connectome import Brain

brain = Brain()
print(f"{brain.n:,} neurons, {brain.n_edges:,} connected pairs, {int(brain.syn.abs().double().sum()):,} synapses")
brain.meta[["bodyId", "type", "instance", "superclass", "side", "transmitter"]].head()

## 2. Finding neurons

`brain.select(...)` returns the rows of the neurons that pass every filter: `type`, `instance`, `side`,
`superclass`, `class_`, `transmitter`, `region`, and `near=(centre, radius_um)` for a ball around a soma.
A name that matches nothing raises an error with the closest names. Look rows up with `brain.meta.loc[rows]`.

In [ ]:
from brain import stimuli

sugar_types = stimuli.shiu_groups()["sugar"]  # Shiu's labellar sugar taste neurons (GRNs), as male-CNS types
sugar = {side: brain.select(type=sugar_types, side=side) for side in "LR"}
MN9 = {side: int(brain.select(instance=f"MN9_{side}")[0]) for side in "LR"}
dns = brain.select(superclass="descending_neuron")
print("sugar GRN types", sugar_types, "->", {side: len(rows) for side, rows in sugar.items()}, "neurons per side")
print(len(dns), "descending neurons")
brain.meta.loc[list(MN9.values()), ["bodyId", "type", "instance", "side", "transmitter", "synapses_in"]]

In [ ]:
# A virtual electrode: every neuron whose soma lies within 30 um of MN9_L's
around = brain.select(near=("MN9_L", 30.0))
brain.meta.loc[around, "type"].value_counts().head(10)

## 3. Shiu's feeding experiment

Drive one side's sugar GRNs with random (Poisson) spikes at 0 to 200 Hz for 1 s and count MN9's spikes. Every
condition runs in its own batch of independent flies, all at once. The synaptic weight is `W_SYN`, Shiu's
0.275 mV scaled by 0.47 for the male CNS (`runs/calibrate_wsyn.py`).

What to expect, from `results/calibrate_wsyn.json`: right-side sugar drives MN9_L to about 18 Hz at 100 Hz and
64 Hz at 200 Hz; left-side sugar drives MN9_L only to about 5 Hz at 200 Hz; MN9_R stays silent. The dashed line
is Shiu's own FlyWire model rerun by that script. Why the two sides differ has not been looked into yet.

In [ ]:
from brain.engine import W_SYN, Engine

RATES = np.arange(0, 201, 20, dtype=np.float32) if GPU else np.array([0, 100, 200], np.float32)
FLIES = 8 if GPU else 1  # independent flies per condition
SHIU_FLYWIRE = [0.0, 0.0, 2.4, 29.5, 50.5, 62.6, 68.8, 77.1, 82.3, 84.7, 91.9]  # MN9 Hz at 0, 20, ..., 200 Hz

inputs = np.concatenate([sugar["L"], sugar["R"]])
conditions = [(side, hz) for side in "LR" for hz in RATES]
rates = np.zeros((len(conditions), len(inputs)), np.float32)
for k, (side, hz) in enumerate(conditions):
    rates[k, np.isin(inputs, sugar[side])] = hz

eng = Engine(brain, len(conditions) * FLIES, inputs, params={"w_syn": W_SYN}, device=DEVICE, seed=0)
eng.drive(np.repeat(rates, FLIES, axis=0))
eng.run(1000.0)  # 1 s of brain time: spike counts are rates in Hz
counts = eng.spikes()[:, [MN9["L"], MN9["R"]]].float().cpu().numpy()
mn9_hz = counts.reshape(2, len(RATES), FLIES, 2).mean(2)  # (driven side, rate, MN9 L/R)
print(f"{len(conditions) * FLIES} flies x 1 s of brain time took {eng.seconds:.0f} s")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, (i, side) in zip(axes, enumerate("LR")):
    for j, mn in enumerate("LR"):
        ax.plot(RATES, mn9_hz[i, :, j], "o-", label=f"MN9_{mn}")
    ax.plot(np.arange(0, 201, 20), SHIU_FLYWIRE, "k--", lw=1, label="Shiu, FlyWire")
    ax.set(title=f"{side} sugar GRNs driven", xlabel="GRN drive (Hz)")
axes[0].set_ylabel("MN9 rate (Hz)")
axes[1].legend()
plt.show()

## 4. Silence one cell type

Shiu's silencing: a silenced neuron still fires, but its synapses have no effect. Find the cell type that
receives the most synapses from the driven sugar GRNs, silence all its neurons, and drive the sugar GRNs again
at 200 Hz. If MN9 drops, that type carries the signal; if not, the signal has other routes. Try other types
from the table.

In [ ]:
side = "LR"[int(mn9_hz[1, -1].max() > mn9_hz[0, -1].max())]  # the side that drives MN9 harder
pre, post, syn = brain.edges()
excite = np.isin(pre, sugar[side]) & (syn > 0)
into = pd.Series(syn[excite], index=post[excite]).groupby(level=0).sum()
by_type = into.groupby(brain.meta.loc[into.index, "type"].to_numpy()).sum().drop(sugar_types, errors="ignore")
by_type = by_type.sort_values(ascending=False)
print(f"cell types most targeted by the {side} sugar GRNs (synapses):")
by_type.head(8)

In [ ]:
top = by_type.index[0]
silenced = brain.select(type=top)


def mn9_at(hz, silence=()):
    eng = Engine(brain, FLIES, sugar[side], params={"w_syn": W_SYN}, device=DEVICE, seed=1, silence=silence)
    eng.drive(np.full(len(sugar[side]), hz, np.float32))
    eng.run(1000.0)
    return eng.spikes()[:, [MN9["L"], MN9["R"]]].float().mean(0).cpu().numpy()


intact, without = mn9_at(200.0), mn9_at(200.0, silenced)
print(f"{side} sugar GRNs at 200 Hz -> MN9 L, R: {intact.round(1)} Hz")
print(f"with all {len(silenced)} {top} neurons silenced: {without.round(1)} Hz")

## 5. A spike raster

One fly, 300 ms, sugar GRNs at 200 Hz (at most 30 neurons of the silenced type are shown). `Engine.step()`
advances 0.1 ms and returns the neurons that fired.

In [ ]:
groups = {"sugar GRNs": sugar[side], top: silenced[:30], "MN9": np.array([MN9["L"], MN9["R"]])}
shown = np.concatenate(list(groups.values()))
eng = Engine(brain, 1, sugar[side], params={"w_syn": W_SYN}, device=DEVICE, seed=2)
eng.drive(np.full(len(sugar[side]), 200.0, np.float32))
line = torch.full((brain.n,), -1, dtype=torch.long, device=eng.dev)  # neuron -> line of the raster
line[torch.as_tensor(shown, device=eng.dev)] = torch.arange(len(shown), device=eng.dev)
times, lines = [], []
for step in range(3000):
    fired = line[eng.step()]  # with one fly, a spike's index is the neuron's row
    fired = fired[fired >= 0].tolist()
    times += [step * eng.p["dt"]] * len(fired)
    lines += fired

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(times, lines, marker="|", s=30, color="k")
edges = np.cumsum([0] + [len(g) for g in groups.values()])
ax.set_yticks((edges[:-1] + edges[1:] - 1) / 2, list(groups))
for e in edges[1:-1]:
    ax.axhline(e - 0.5, color="0.8", lw=0.8)
ax.set(xlabel="time (ms)", xlim=(0, 300), ylim=(-1, len(shown)))
plt.show()

## Where next

- [docs/ONBOARDING.md](https://github.com/victory-c/fly-brain/blob/main/docs/ONBOARDING.md): setup on a laptop or
  the OCF cluster, the reading pack, how to send changes.
- [docs/CURRENT_NUMBERS.md](https://github.com/victory-c/fly-brain/blob/main/docs/CURRENT_NUMBERS.md): which
  results in the repo come from the current simulator.
- Things to try here: drive `stimuli.group("bitter")` together with sugar; change the weight scale
  (`params={"w_syn": ...}`); record the descending neurons (`dns`) instead of MN9; select neurons `near=` a
  point of your choice.